# Binary ASD vs TD — WavLM-Base-Plus deep-learning pipeline

**This is the binary variant: ADHD is removed entirely and the task is two-class, ASD vs TD.**
Derived from the 3-class DL-only notebook; the classical-ML sections (Table 1, Table 2) were already
removed upstream.

# Based on "Evaluating Voice Biomarkers and Deep Learning for Neurodevelopmental Disorder Screening in Real-World Conditions"

Rakotomanana & Rouhafzay (2025), *Eng. Proc.* 2025, 118, 46. https://doi.org/10.3390/ECSA-12-26523

The base paper classifies **three** groups (ASD / ADHD / TD). This notebook deliberately narrows that
to **two** (ASD / TD) on the SK sub-corpus of the Asymmetries Project (TalkBank/CHILDES), keeping the
paper's WavLM-Base-Plus architecture, preprocessing and training recipe unchanged.

### What removing ADHD changes, and what it does not

| | 3-class (paper) | This notebook (binary) |
|---|---|---|
| Classes | ASD, ADHD, TD | **ASD, TD** |
| Chance accuracy | 33.3% | **50.0%** |
| `N_CLASSES` / output layer | 3 | **2** |
| Comparable to the paper's 0.769? | yes | **no — different task** |

The last row is the one that matters for the thesis write-up. An accuracy of, say, 0.78 here is *not*
"matching the paper" — it is a two-way choice against the paper's three-way one, and an easier problem
on its face. Section 6 and Section 9 below label the paper's number as a reference point, never as a
like-for-like baseline, and report accuracy against the 50% chance line.

Because the task is binary, Section 6 also reports **sensitivity and specificity** with ASD as the
positive class — the pair a screening instrument is actually judged on, and which has no single-number
equivalent in the 3-class setting.

**Dataset layout expected** (adjust `DATA_ROOT` in the config cell if yours differs):

```
/kaggle/input/<dataset-name>/dataset/
├── Asymmetries/            # .cha CHAT transcripts (child-speech timestamps)
│   ├── SK-ASD/   asd01.cha  ...
│   └── SK-TD/    td01.cha   ... td38.cha
├── SK-ASD/                  # matching audio, same basenames
│   ├── asd01.mp3 ...
└── SK-TD/
    └── td01.mp3 ...
```

An `SK-ADHD` folder may sit alongside these — it is ignored. Section 2 asserts that exactly ASD and TD
are active, so a stray ADHD folder cannot quietly turn this back into a 3-class run.

> **Known deviations from the paper to keep in mind when comparing numbers:**
> - The task itself is binary here, not 3-class — see the table above.
> - Audio here is `.mp3`, not the original Olympus `.wma` — re-encoding/lossy compression can shift
>   jitter/shimmer/HNR slightly.
> - The paper doesn't specify the exact WavLM layer used for "mid-layer" pooling — this notebook exposes
>   it as a hyperparameter (`WAVLM_LAYER`) to sweep.
> - The paper used a single stratified split for the deep model; `USE_CV=True` here runs 5-fold
>   participant-level CV instead, which is stricter.


## 0. Setup

In [ ]:
# Install packages not preinstalled on Kaggle (safe to re-run)
!pip install -q praat-parselmouth pylangacq soundfile librosa audiomentations


In [ ]:
import os, re, json, math, random, warnings, itertools
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

import librosa
import soundfile as sf
import parselmouth
from parselmouth.praat import call as praat_call

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Environment ready.")


## 0b. GPU verification

Kaggle notebooks default to **CPU only** — this has to be turned on by hand, this file can't do it for you:
open **Settings** (the ⋮ menu / gear icon in the right sidebar) → **Accelerator** → pick **GPU T4 x2** (or
**GPU P100**) → **Save**, then re-run. The cell below confirms the accelerator actually took effect before
any time is spent on preprocessing; Sections 1-7 run fine on CPU, but Section 8's WavLM fine-tuning (25
epochs) is impractical without a GPU.

In [ ]:
import subprocess
import torch

cuda_ok = torch.cuda.is_available()
print(f"PyTorch: {torch.__version__} | CUDA available: {cuda_ok}")

if cuda_ok:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB total")
    try:
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu",
             "--format=csv"],
            capture_output=True, text=True, check=True,
        )
        print(smi.stdout)
    except Exception as e:
        print(f"  (nvidia-smi unavailable: {e})")
else:
    print(
        "\n*** No GPU detected — accelerator is off (or set to 'None'). ***\n"
        "On Kaggle: Settings (⋮ menu, top right) -> Accelerator -> 'GPU T4 x2' or 'GPU P100' -> Save,\n"
        "then Run All again. Sections 1-7 (conventional acoustic-feature pipeline) still work on CPU,\n"
        "but Section 8 (WavLM-Base-Plus fine-tuning, 25 epochs) will be impractically slow without a GPU\n"
        "and cell-26 below will raise once training is about to start."
    )


## 1. Configuration\n\nAdjust `DATA_ROOT` to match your Kaggle input mount path.

In [ ]:
# ---- EDIT THIS to match your Kaggle "Add Input" mount path ----
DATA_ROOT = Path("/kaggle/input/datasets/asifshafin/asd-voice-full")   # <- change if your dataset slug differs
# (CHA_ROOT / AUDIO_ROOT are gone: the old config joined them with absolute paths, where the
# absolute operand wins and the prefix is silently discarded. Paths are built from _DATASET below.)

WORK_DIR = Path("/kaggle/working")
SEGMENTS_DIR = WORK_DIR / "segments"          # cropped child-speech clips get written here
FEATURES_CSV = WORK_DIR / "acoustic_features.csv"
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

# class label -> (cha subfolder name, audio subfolder name, filename prefix)
# BINARY TASK: ASD vs TD only. ADHD is deliberately absent — not commented out, removed, so that
# there is no "option" left to half-enable. Everything downstream (N_CLASSES, the output layer, the
# class weights, the confusion matrix) is derived from what is in this dict, so this is the single
# place the task's class set is defined.
_DATASET = Path("/kaggle/input/datasets/asifshafin/asd-voice-full/dataset")

CLASS_CONFIG = {
    "ASD": {"cha_dir": _DATASET / "/kaggle/input/datasets/asifshafin/asd-voice-full/dataset/Asymmetries/SK-ASD", "audio_dir": _DATASET / "/kaggle/input/datasets/asifshafin/asd-voice-full/dataset/SK-ASD", "prefix": "asd"},
    "TD":  {"cha_dir": _DATASET / "/kaggle/input/datasets/asifshafin/asd-voice-full/dataset/Asymmetries/SK-TD",  "audio_dir": _DATASET / "/kaggle/input/datasets/asifshafin/asd-voice-full/dataset/SK-TD",  "prefix": "td"},
}

# What Section 2 will enforce. If an SK-ADHD folder exists in the dataset it is simply never looked
# at, and this assertion makes sure it cannot creep back in through an edit above.
EXPECTED_CLASSES = ["ASD", "TD"]

MIN_SEGMENT_SEC = 1.0     # paper: exclude child-speech segments shorter than 1s
TARGET_SR = 16000         # WavLM input rate; also used for feature extraction for consistency


## 2. Discover available classes & files

Confirms that both class folders have matched `.cha` + audio pairs, then **asserts the active class
set is exactly ASD and TD**. A missing folder or an unexpected extra class stops the run here rather
than changing the task shape silently three sections later.


In [ ]:
def discover_pairs(class_name, cfg):
    cha_dir, audio_dir, prefix = cfg["cha_dir"], cfg["audio_dir"], cfg["prefix"]
    if not cha_dir.exists() or not audio_dir.exists():
        return []
    cha_files = {p.stem: p for p in cha_dir.glob("*.cha")}
    audio_exts = (".mp3", ".wav", ".wma", ".m4a")
    audio_files = {p.stem: p for p in audio_dir.iterdir() if p.suffix.lower() in audio_exts}
    pairs = []
    for stem, cha_path in sorted(cha_files.items()):
        if stem in audio_files:
            pairs.append((stem, cha_path, audio_files[stem]))
        else:
            print(f"  [warn] {class_name}: no audio match for {cha_path.name}")
    return pairs

ACTIVE_CLASSES = {}
for cls, cfg in CLASS_CONFIG.items():
    pairs = discover_pairs(cls, cfg)
    if pairs:
        ACTIVE_CLASSES[cls] = pairs
        print(f"{cls}: {len(pairs)} matched participant(s)")
    else:
        print(f"{cls}: 0 files found — skipped (folder missing or empty)")

# Binary task: fail loudly rather than quietly training a differently-shaped model.
found = sorted(ACTIVE_CLASSES.keys())
if found != sorted(EXPECTED_CLASSES):
    raise ValueError(
        f"Expected exactly {sorted(EXPECTED_CLASSES)} but found {found}.\n"
        f"  - missing a class -> check the folder paths in CLASS_CONFIG above\n"
        f"  - an extra class  -> this notebook is the BINARY ASD-vs-TD variant; use the 3-class "
        f"notebook instead of adding a class here."
    )

N_CLASSES = len(ACTIVE_CLASSES)
print(f"\nBinary mode confirmed: {found}  (chance accuracy = {1 / N_CLASSES:.1%})")
print("ADHD is not part of this task — an SK-ADHD folder, if present, is ignored.")


## 3. Exploratory data analysis — participant counts & demographics

Reproduces the demographic summary style of the paper's Section 3.1 (participant counts; age/sex if
present in the `.cha` `@ID` header lines).


In [ ]:
ID_LINE_RE = re.compile(r"^@ID:\s*(.*)$")

def parse_id_headers(cha_path):
    """Parse CHAT @ID header lines. Returns list of dicts (one per participant/tier declared)."""
    records = []
    with open(cha_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = ID_LINE_RE.match(line.strip())
            if not m:
                continue
            fields = m.group(1).split("|")
            # CHAT @ID order: language|corpus|code|age|sex|group|SES|role|education|custom
            while len(fields) < 9:
                fields.append("")
            records.append({
                "language": fields[0], "corpus": fields[1], "code": fields[2],
                "age": fields[3], "sex": fields[4], "group": fields[5],
                "role": fields[7],
            })
    return records

def child_record(cha_path):
    for rec in parse_id_headers(cha_path):
        if "Target_Child" in rec["role"] or rec["role"] == "Child":
            return rec
    return None

demo_rows = []
for cls, pairs in ACTIVE_CLASSES.items():
    for stem, cha_path, audio_path in pairs:
        rec = child_record(cha_path)
        demo_rows.append({
            "class": cls, "participant": stem,
            "age_raw": rec["age"] if rec else None,
            "sex": rec["sex"] if rec else None,
        })
demo_df = pd.DataFrame(demo_rows)

def parse_chat_age_to_years(age_str):
    # CHAT age format: e.g. "9;3.15" = 9 years 3 months 15 days
    if not age_str:
        return np.nan
    m = re.match(r"(\d+);(\d+)?", age_str)
    if not m:
        return np.nan
    years = int(m.group(1))
    months = int(m.group(2)) if m.group(2) else 0
    return years + months / 12

demo_df["age_years"] = demo_df["age_raw"].apply(parse_chat_age_to_years)

summary = demo_df.groupby("class").agg(
    n=("participant", "count"),
    mean_age=("age_years", "mean"),
    pct_male=("sex", lambda s: (s.str.upper() == "M").mean() * 100 if s.notna().any() else np.nan),
).round(2)
display(summary)

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=summary.index, y="n", data=summary.reset_index(), ax=ax, palette="Set2")
ax.set_title("Participants per class")
ax.set_ylabel("N children")
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_class_counts.png", dpi=150)
plt.show()


## 4. Preprocessing — crop child-only speech from `.cha` timestamps

CHAT files mark utterance-level timing with a bullet code `\x15start_end\x15` (milliseconds) appended
to the speaker's tier line. We isolate the child speaker's tier (role == Target_Child, tier code found
from the `@ID` header, typically `CHI`), collect its timestamped intervals, and crop those spans out of
the full session audio — matching the paper's Section 3.2.


In [ ]:
TIER_RE = re.compile(r"^\*([A-Za-z0-9_]+):\t?(.*)$")
BULLET_TS_RE = re.compile(r"\x15(\d+)_(\d+)\x15")

def get_child_tier_code(cha_path):
    for rec in parse_id_headers(cha_path):
        if "Target_Child" in rec["role"] or rec["role"] == "Child":
            return rec["code"] or "CHI"
    return "CHI"  # CHAT convention fallback

def extract_child_intervals_ms(cha_path):
    """Return list of (start_ms, end_ms) for the child speaker's utterances."""
    child_code = get_child_tier_code(cha_path)
    intervals = []
    current_tier = None
    buf = ""
    with open(cha_path, encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    for line in lines:
        line = line.rstrip("\n")
        m = TIER_RE.match(line)
        if m:
            # flush previous buffered utterance
            if current_tier == child_code:
                for s, e in BULLET_TS_RE.findall(buf):
                    intervals.append((int(s), int(e)))
            current_tier, buf = m.group(1), m.group(2)
        elif line.startswith("%"):
            # dependent tier (e.g. %mor, %gra) - not part of the utterance text
            continue
        elif line.startswith("@"):
            if current_tier == child_code:
                for s, e in BULLET_TS_RE.findall(buf):
                    intervals.append((int(s), int(e)))
            current_tier, buf = None, ""
        else:
            # continuation of a wrapped utterance line
            if current_tier is not None:
                buf += " " + line
    if current_tier == child_code:
        for s, e in BULLET_TS_RE.findall(buf):
            intervals.append((int(s), int(e)))
    return intervals

def crop_child_segments(class_name, stem, cha_path, audio_path, out_dir):
    intervals = extract_child_intervals_ms(cha_path)
    if not intervals:
        return []
    y, sr = librosa.load(str(audio_path), sr=TARGET_SR, mono=True)
    out_paths = []
    for i, (s_ms, e_ms) in enumerate(intervals):
        if e_ms <= s_ms:
            continue
        dur_s = (e_ms - s_ms) / 1000.0
        if dur_s < MIN_SEGMENT_SEC:
            continue
        s_samp, e_samp = int(s_ms / 1000 * sr), int(e_ms / 1000 * sr)
        e_samp = min(e_samp, len(y))
        if e_samp <= s_samp:
            continue
        clip = y[s_samp:e_samp]
        out_path = out_dir / f"{stem}_{i:03d}.wav"
        sf.write(out_path, clip, sr)
        out_paths.append(out_path)
    return out_paths

# Run cropping for every participant in every active class
segment_index = []  # rows: class, participant, segment_path, duration_s
for cls, pairs in ACTIVE_CLASSES.items():
    cls_dir = SEGMENTS_DIR / cls
    cls_dir.mkdir(parents=True, exist_ok=True)
    for stem, cha_path, audio_path in pairs:
        seg_paths = crop_child_segments(cls, stem, cha_path, audio_path, cls_dir)
        for sp in seg_paths:
            dur = librosa.get_duration(path=str(sp))
            segment_index.append({"class": cls, "participant": stem, "segment_path": str(sp), "duration_s": dur})
        if not seg_paths:
            print(f"  [warn] no valid (>= {MIN_SEGMENT_SEC}s) child segments extracted for {cls}/{stem} "
                  f"— check .cha bullet-timestamp coverage for this file")

seg_df = pd.DataFrame(segment_index)
print(f"Total child-speech segments extracted: {len(seg_df)}")
seg_df.groupby("class").agg(n_segments=("segment_path", "count"),
                             n_participants=("participant", "nunique"),
                             mean_dur_s=("duration_s", "mean")).round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(data=seg_df, x="duration_s", hue="class", bins=30, element="step", stat="density", common_norm=False, ax=ax)
ax.set_title("Child-speech segment duration distribution by class")
ax.set_xlabel("Duration (s)")
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_segment_durations.png", dpi=150)
plt.show()


## 5. Deep learning pipeline — WavLM-Base-Plus (reproduces Table 3, Figure 1 framework)

Fine-tunes `microsoft/wavlm-base-plus` with masked mean pooling over a configurable mid-layer's hidden
states, a dropout + linear classification head, weighted cross-entropy with label smoothing, and the
paper's augmentation recipe (speed perturbation, time dropout, gain jitter).

In [ ]:
!pip install -q transformers accelerate


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# ^ these lived in the classical-ML section that was removed to build this DL-only notebook — Section 6
#   (test eval) and Section 9 (summary) below still need them, hence the NameError you hit.

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is off for this session (checked again here since Section 5 is the expensive part). "
        "Enable it via Settings -> Accelerator -> 'GPU T4 x2' or 'GPU P100' -> Save, then re-run from "
        "Section 0b onward. Fine-tuning WavLM-Base-Plus for 25 epochs on CPU is not practical."
    )
DEVICE = torch.device("cuda")
print(f"Using device: {DEVICE} ({torch.cuda.get_device_name(0)})")

WAVLM_CHECKPOINT = "microsoft/wavlm-base-plus"
WAVLM_LAYER = 6          # mid-layer index to pool from — sweep this if reproducing exactly
MAX_DURATION_S = 10
MIN_DURATION_S = 1.0
BATCH_SIZE = 8
N_EPOCHS = 25
LABEL_SMOOTHING = 0.05
N_UNFROZEN_LAYERS = 2    # only fine-tune the last N encoder layers that feed the pooled output + head
HEAD_LR = 1e-3           # the randomly-initialised head needs a much larger LR than the pretrained
                         # encoder layers do — sharing LR with them leaves it barely moving

# ===========================================================================
# REGULARISATION -- tuned against overfitting at N_UNFROZEN_LAYERS = 2
# ===========================================================================
# Read the markdown cell before the model definition for the reasoning. The short version of
# what changed and why:
#
# LR is THE lever at this setting. With N_UNFROZEN_LAYERS = 2 and WAVLM_LAYER = 6, exactly two
# pretrained layers train (4 and 5). LR touches only those two -- the head has its own HEAD_LR --
# so halving it is a targeted constraint on how far the pretrained weights may drift, which is
# what actually regularises a fine-tune. It does NOT slow the head down.
LR = 5e-6                # was 1e-5. Halved. Raise back to 1e-5 if TRAINING accuracy underfits.

DROPOUT = 0.2            # was 0.1 (and was hardcoded -- changing it had no effect at all, see
                         # the model cell). The most direct knob you have. Try 0.3 if needed.

WEIGHT_DECAY = 0.01      # AdamW decay. Biases and LayerNorm gains are now EXEMPT -- previously
                         # AdamW's 0.01 default hit every parameter including those, which shrinks
                         # activations rather than regularising. Raise to 0.05 for more.

LAYERWISE_LR_DECAY = 0.9 # per-layer LR multiplier going down the encoder, measured from the
                         # POOLING layer (layers above it never contribute). Honest caveat: with
                         # only 2 trainable layers this spreads them by just 1.11x, so it is a
                         # minor effect here -- it matters if you raise N_UNFROZEN_LAYERS.
                         # 1.0 = off.

EARLY_STOP_PATIENCE = 5  # stop a fold after this many epochs with no val-loss improvement.
                         # This does not change the reported number -- the best-val-loss
                         # checkpoint is what gets evaluated either way -- it just stops paying
                         # for epochs spent overfitting. 0 disables it.

# seg_df (built in Section 4) is the earliest point that has the class labels.
#
# Section 2 set N_CLASSES from the FOLDERS discovered; this list comes from the SEGMENTS actually
# extracted. They can disagree — a class whose .cha files yield no usable segment disappears here
# while still counting there — and the model's output layer is built from N_CLASSES while its labels
# come from this list, so a disagreement means an output unit no label ever selects. Check it.
CLASSES_SORTED = sorted(seg_df["class"].unique())
if CLASSES_SORTED != sorted(EXPECTED_CLASSES):
    raise ValueError(
        f"Segments cover {CLASSES_SORTED} but the task is {sorted(EXPECTED_CLASSES)}. "
        f"A class discovered in Section 2 produced no usable segments in Section 4 — check the "
        f"[warn] lines there for a participant whose .cha has no child-speech timestamps."
    )
N_CLASSES = len(CLASSES_SORTED)   # re-derived from the labels the model will actually see
print(f"Classes: {CLASSES_SORTED}  (binary, chance = {1 / N_CLASSES:.1%})")

label2id = {c: i for i, c in enumerate(CLASSES_SORTED)}
id2label = {i: c for c, i in label2id.items()}

print(f"Regularisation: N_UNFROZEN_LAYERS={N_UNFROZEN_LAYERS} | LR={LR:g} (head {HEAD_LR:g}) | "
      f"dropout={DROPOUT} | wd={WEIGHT_DECAY} | LLRD={LAYERWISE_LR_DECAY} | "
      f"early stop patience={EARLY_STOP_PATIENCE}")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_CHECKPOINT)


In [ ]:
# Evaluation protocol: single held-out split (the paper's DL setup) or 5-fold CV.
#
# The paper used 10-fold nested CV for the CLASSICAL classifiers (Tables 1-2) but only a single
# stratified train/val/test split for the deep model (Table 3). USE_CV=True is therefore a
# deliberate deviation — a stricter evaluation, since a single test fold here is only ~18 of the
# 121 participants and one child landing differently moves accuracy by several points.
USE_CV = True
N_FOLDS = 5
VAL_FRACTION = 0.15   # carved out of each fold's TRAINING participants, for checkpoint selection

from sklearn.model_selection import StratifiedKFold

# One row per participant. EVERY split below is made on this table, never on segments: a child's
# clips are near-duplicates of each other, so splitting by segment would put the same voice in
# train and test and inflate the score badly.
participants_df = seg_df.drop_duplicates("participant")[["participant", "class"]].reset_index(drop=True)
print(f"Participants: {len(participants_df)} total")
print(participants_df["class"].value_counts().to_string())

def subset_by_participants(df, part_df):
    return df[df["participant"].isin(part_df["participant"])].reset_index(drop=True)

def describe(name, tr, va, te):
    print(f"  {name}: train {len(tr):>5} seg / {tr['participant'].nunique():>3} kids | "
          f"val {len(va):>4} seg / {va['participant'].nunique():>2} kids | "
          f"test {len(te):>4} seg / {te['participant'].nunique():>2} kids")

if USE_CV:
    smallest = participants_df["class"].value_counts().min()
    if smallest < N_FOLDS:
        raise ValueError(
            f"N_FOLDS={N_FOLDS} but the smallest class has only {smallest} participants. "
            f"Stratified folds need at least one participant per class per fold — lower N_FOLDS "
            f"to {smallest} or less, or set USE_CV=False."
        )
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    FOLDS = []
    for fold, (fit_idx, test_idx) in enumerate(
            skf.split(participants_df, participants_df["class"]), start=1):
        fit_p = participants_df.iloc[fit_idx]
        test_p = participants_df.iloc[test_idx]
        # carve a validation set out of this fold's training participants (never out of test)
        tr_p, va_p = train_test_split(
            fit_p, test_size=VAL_FRACTION, stratify=fit_p["class"], random_state=RANDOM_SEED)
        FOLDS.append({
            "fold": fold,
            "train_df": subset_by_participants(seg_df, tr_p),
            "val_df": subset_by_participants(seg_df, va_p),
            "test_df": subset_by_participants(seg_df, test_p),
        })

    print(f"\n{N_FOLDS}-fold participant-level CV — every participant is tested exactly once:")
    for f in FOLDS:
        describe(f"fold {f['fold']}", f["train_df"], f["val_df"], f["test_df"])

    # sanity: no participant may appear in two different test folds, and none may be in both
    # train and test within a fold
    seen = set()
    for f in FOLDS:
        te = set(f["test_df"]["participant"])
        assert not (seen & te), "a participant landed in two test folds"
        seen |= te
        assert not (set(f["train_df"]["participant"]) & te), "participant leaked train->test"
        assert not (set(f["val_df"]["participant"]) & te), "participant leaked val->test"
    assert seen == set(participants_df["participant"]), "not every participant was tested"
    print(f"  leakage checks passed; {len(seen)} participants each tested exactly once")
else:
    train_p, temp_p = train_test_split(
        participants_df, test_size=0.3, stratify=participants_df["class"], random_state=RANDOM_SEED)
    val_p, test_p = train_test_split(
        temp_p, test_size=0.5, stratify=temp_p["class"], random_state=RANDOM_SEED)
    FOLDS = [{
        "fold": 1,
        "train_df": subset_by_participants(seg_df, train_p),
        "val_df": subset_by_participants(seg_df, val_p),
        "test_df": subset_by_participants(seg_df, test_p),
    }]
    print("\nSingle held-out split (paper's DL protocol):")
    describe("split", FOLDS[0]["train_df"], FOLDS[0]["val_df"], FOLDS[0]["test_df"])

# kept so any cell that still refers to the single-split names works
train_df, val_df, test_df = FOLDS[0]["train_df"], FOLDS[0]["val_df"], FOLDS[0]["test_df"]

In [ ]:
def speed_perturb(y, sr, factor_range=(0.9, 1.1)):
    factor = np.random.uniform(*factor_range)
    return librosa.effects.time_stretch(y, rate=factor)

def gain_jitter(y, db_range=(-6, 6)):
    gain_db = np.random.uniform(*db_range)
    return y * (10 ** (gain_db / 20))

def time_dropout(y, sr, max_drop_s=0.5, n_drops=2):
    y = y.copy()
    for _ in range(n_drops):
        drop_len = int(np.random.uniform(0, max_drop_s) * sr)
        if drop_len == 0 or len(y) <= drop_len:
            continue
        start = np.random.randint(0, len(y) - drop_len)
        y[start:start + drop_len] = 0.0
    return y

def pad_or_crop(y, sr, target_s=MAX_DURATION_S, random_crop=True):
    """Returns (fixed_length_array, valid_length).

    valid_length = how much of the array is real audio rather than zero-padding, so pooling can
    exclude the padding instead of averaging it in. With MAX_DURATION_S=7.5 and mean clip lengths
    that differ by class (TD clips run shorter than ASD ones in this corpus), unmasked pooling
    dilutes the shorter class more — a class-dependent handicap, not a neutral one.

    random_crop=False makes the crop deterministic. Inference MUST use that: predict_proba is
    called twice per occlusion measurement (clean vs occluded), and a random window each call
    means the difference measures which window was drawn, not the occlusion."""
    target_len = int(target_s * sr)
    if len(y) >= target_len:
        if random_crop and len(y) > target_len:
            start = np.random.randint(0, len(y) - target_len + 1)
        else:
            start = 0
        return y[start:start + target_len], target_len
    return np.pad(y, (0, target_len - len(y))), len(y)

class SpeechDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y, sr = librosa.load(row["segment_path"], sr=TARGET_SR, mono=True)
        if len(y) < int(MIN_DURATION_S * sr):
            y = np.pad(y, (0, int(MIN_DURATION_S * sr) - len(y)))
        if self.augment:
            if np.random.rand() < 0.5:
                y = speed_perturb(y, sr)
            if np.random.rand() < 0.5:
                y = gain_jitter(y)
            if np.random.rand() < 0.5:
                y = time_dropout(y, sr)
        y, valid_len = pad_or_crop(y, sr, random_crop=self.augment)
        label = label2id[row["class"]]
        return {
            "input_values": torch.tensor(y, dtype=torch.float32),
            "valid_length": valid_len,
            "label": label,
        }

def collate_fn(batch):
    input_values = torch.stack([b["input_values"] for b in batch])
    lengths = torch.tensor([b["valid_length"] for b in batch], dtype=torch.long)
    # 1 where the sample is real audio, 0 where it is padding. WavLM's own
    # _get_feature_vector_attention_mask (used in the model) downsamples this to match the
    # encoder's time axis, so masked pooling averages over real speech only.
    attention_mask = (torch.arange(input_values.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)).long()
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    return {"input_values": input_values, "attention_mask": attention_mask, "labels": labels}

def make_loaders(train_df, val_df, test_df):
    """Built per fold rather than once, so each fold gets loaders over its own participants."""
    train_ds = SpeechDataset(train_df, augment=True)
    val_ds = SpeechDataset(val_df, augment=False)
    test_ds = SpeechDataset(test_df, augment=False)
    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn),
        DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn),
        DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn),
    )


### Controlling overfitting at `N_UNFROZEN_LAYERS = 2`

You have **84 participants**. The 2707 segments do not change that — clips from the same child are
near-duplicates, so the effective sample size is 84. This section keeps your original partial-freeze
setting and tightens everything else around it.

**Which knob actually matters here, and why it is `LR`.** With `N_UNFROZEN_LAYERS = 2` and
`WAVLM_LAYER = 6`, exactly two pretrained layers train: 4 and 5. `LR` touches *only those two* —
the classifier head has its own `HEAD_LR = 1e-3`. So halving `LR` to **5e-6** constrains how far the
pretrained weights may drift without slowing the head's learning at all.

That is the mechanism that matters. A lower step size does not reduce capacity — with enough epochs
you reach the same overfit minimum, just slower. What a low LR *does* do when fine-tuning a
pretrained model is keep the weights near their pretrained values, and the pretrained solution acts
as a prior. Constraining drift is real regularisation; constraining speed is not.

**An honest note on layer-wise LR decay.** It is wired in, and the reference point is now the
*pooling* layer rather than layer 11 (layers above the pooling point receive no gradient, so
counting them would have pushed layers 4 and 5 down to roughly half LR for no reason). But with only
two trainable layers it spreads them by a factor of **1.11** — a minor effect. LLRD earns its keep
when many layers are trainable. Here, `LR` does the work.

**What changed, and what each is worth:**

| | was | now | effect |
|---|---|---|---|
| `LR` | 1e-5 | **5e-6** | the main lever — halves drift on layers 4 and 5 |
| `DROPOUT` | 0.1, **and had no effect at all** | **0.2**, actually wired in | second lever |
| weight decay | 0.01 on **every** parameter | 0.01, biases/LayerNorm **exempt** | fixes real damage |
| `LAYERWISE_LR_DECAY` | — | 0.9, from the pooling layer | minor at 2 layers |
| `EARLY_STOP_PATIENCE` | — | 5 | saves time, not accuracy |

**Read the curves before turning anything further.** The training-curve cell plots train vs val:

| what you see | meaning | do |
|---|---|---|
| Val loss bottoms out then rises; val accuracy holds | Normal. The **best-val-loss checkpoint is what gets evaluated**, so this costs time, not accuracy | Nothing |
| Val accuracy peaks then falls well below its best | Real overfitting | `DROPOUT = 0.3`, then `WEIGHT_DECAY = 0.05`, then `LR = 2e-6` |
| Train accuracy stuck below ~0.75 | **Underfitting** — you cut too far | `LR = 1e-5`, `DROPOUT = 0.1` |
| Train and val both near chance | Not overfitting — not learning | Check labels and that the loss is finite |

**Why early stopping does not change your result.** The fold already evaluates the best-val-loss
checkpoint, not the last epoch. Stopping early just avoids paying for epochs after that point. With
5 folds × 25 epochs that is real time saved, and nothing else.

**One inefficiency left in place deliberately.** Pooling is from a fixed `WAVLM_LAYER = 6`, so
encoder layers 7–12 are computed on every forward pass and contribute nothing — they cannot even
receive gradient. Deleting them would give roughly a 2x speedup at identical outputs, and replacing
fixed-layer pooling with a learned weighted sum over all 13 hidden states would let the model pick
the depth instead of you guessing. Neither is done here, to keep this change set to regularisation.


In [ ]:
class WavLMClassifier(nn.Module):
    def __init__(self, checkpoint, n_classes, mid_layer=WAVLM_LAYER, dropout=DROPOUT,
                 n_unfrozen_layers=N_UNFROZEN_LAYERS):
        super().__init__()
        self.encoder = WavLMModel.from_pretrained(checkpoint)

        # LayerDrop MUST be off, because this head pools a FIXED layer index.
        # wavlm-base-plus ships layerdrop=0.05. In train() mode the encoder randomly skips layers
        # and does NOT record a hidden state for a skipped one, so output_hidden_states returns a
        # SHORTER tuple and every index above the skipped layer shifts down by one. Pooling
        # hidden_states[6] then silently reads layer 5, or 4, on roughly half of all training
        # batches, while validation and test (eval mode, LayerDrop inactive) always read the true
        # layer 6. The head is trained on one representation and evaluated on another, with no
        # error raised — and with enough simultaneous drops the index goes out of range outright.
        self.encoder.config.layerdrop = 0.0
        self.encoder.encoder.config.layerdrop = 0.0

        self.mid_layer = mid_layer
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, n_classes)
        self._freeze_encoder(n_unfrozen_layers)

    def _freeze_encoder(self, n_unfrozen_layers):
        """Freeze the encoder except the last `n_unfrozen_layers` layers that actually FEED the
        pooled representation (pooling happens at hidden_states[mid_layer], so only encoder layers
        0..mid_layer-1 have any effect on the classifier — layers above that get computed for
        nothing and receive zero gradient no matter what, freezing or not)."""
        layers = self.encoder.encoder.layers
        n_layers = len(layers)
        pool_idx = self.mid_layer if self.mid_layer >= 0 else n_layers + 1 + self.mid_layer
        pool_idx = max(0, min(pool_idx, n_layers))
        n_contributing = pool_idx

        for p in self.encoder.parameters():
            p.requires_grad = False
        first_unfrozen = max(0, n_contributing - n_unfrozen_layers)
        for layer in layers[first_unfrozen:n_contributing]:
            for p in layer.parameters():
                p.requires_grad = True

        n_total = sum(p.numel() for p in self.parameters())
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Pooling from hidden_states[{self.mid_layer}] = output of encoder layer {pool_idx - 1} "
              f"of {n_layers}")
        if n_contributing == 0:
            print("  *** WARNING: pooling happens before any encoder layer — only the classifier "
                  "head can train. Raise WAVLM_LAYER. ***")
        else:
            print(f"  trainable encoder layers: {list(range(first_unfrozen, n_contributing))}")
            print(f"  layers {list(range(n_contributing, n_layers))} are never used by this head "
                  f"— left frozen on purpose")
        print(f"  trainable params: {n_trainable:,} / {n_total:,} ({100 * n_trainable / n_total:.1f}%)")

        # build_param_groups needs to know where the contributing stack ends, so layer-wise LR
        # decay is measured from the POOLING layer rather than from layer 11. Layers above the
        # pooling point receive no gradient at all, so counting them would push the layers that
        # DO train down to roughly half LR for no reason.
        self.pool_depth = n_contributing

    def forward(self, input_values, attention_mask=None):
        outputs = self.encoder(input_values, attention_mask=attention_mask, output_hidden_states=True)
        # One state per layer plus the input embedding. If this ever fails, LayerDrop is back on
        # (see __init__) and the layer index below no longer means what it says.
        expected = self.encoder.config.num_hidden_layers + 1
        if len(outputs.hidden_states) != expected:
            raise RuntimeError(
                f"Got {len(outputs.hidden_states)} hidden states, expected {expected}. LayerDrop is "
                f"active, so hidden_states[{self.mid_layer}] is not layer {self.mid_layer}."
            )
        hidden = outputs.hidden_states[self.mid_layer]           # (B, T, H)
        if attention_mask is not None:
            feat_mask = self.encoder._get_feature_vector_attention_mask(hidden.shape[1], attention_mask)
            mask = feat_mask.unsqueeze(-1).float()
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
        else:
            pooled = hidden.mean(1)
        pooled = self.dropout(pooled)
        return self.classifier(pooled)

def build_param_groups(model):
    """Per-layer learning rates + correct weight-decay exclusions.

    Two things happen here that the previous flat two-group setup did not do.

    1) Layer-wise LR decay. Each trainable encoder layer gets LAYERWISE_LR_DECAY x the LR of the
       one above it, measured from the POOLING layer -- not from the top of the 12-layer stack,
       because layers above the pooling point receive no gradient and counting them would halve
       the LR of the layers that actually train, for no reason.

       Honest note on scale: at N_UNFROZEN_LAYERS = 2 only two layers train, so this spreads them
       by a factor of 1.11 and is a minor effect. It is LR itself that does the work at this
       setting. LLRD earns its keep if you raise N_UNFROZEN_LAYERS.

    2) Weight-decay exclusions. Biases and LayerNorm gains are exempt. The previous code called
       AdamW(param_groups) with no weight_decay argument, which silently applies PyTorch's default
       of 0.01 to every parameter -- biases and LayerNorm gains included. Decaying a LayerNorm
       gain toward zero shrinks the activations it normalises; that is damage, not regularisation.
    """
    NO_DECAY_KEYS = ("bias", "layer_norm", "layernorm")
    pool_depth = getattr(model, "pool_depth", model.encoder.config.num_hidden_layers)
    n_layers = model.encoder.config.num_hidden_layers

    # Where the encoder's final LayerNorm sits depends on the variant, and it is the opposite of
    # what the name suggests. do_stable_layer_norm=True (XLS-R, wav2vec2-large) applies it AFTER
    # the stack; False (wavlm-base-plus) applies it BEFORE, as part of the input path. Read the
    # flag rather than assuming.
    stable = bool(getattr(model.encoder.config, "do_stable_layer_norm", False))

    def depth_of(name):
        if name.startswith("classifier"):
            return None                      # head: uses HEAD_LR, not the decay schedule
        if "encoder.layers." in name:
            return int(name.split("encoder.layers.")[1].split(".")[0]) + 1
        if name.startswith("encoder.encoder.layer_norm"):
            return n_layers if stable else 0
        return 0   # feature_extractor, feature_projection, pos_conv_embed, masked_spec_embed

    buckets = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        d = depth_of(name)
        lr = HEAD_LR if d is None else LR * (LAYERWISE_LR_DECAY ** max(0, pool_depth - d))
        wd = 0.0 if any(k in name.lower() for k in NO_DECAY_KEYS) else WEIGHT_DECAY
        buckets.setdefault((lr, wd), {"params": [], "lr": lr, "weight_decay": wd})
        buckets[(lr, wd)]["params"].append(p)

    if not buckets:
        raise ValueError("No trainable parameters -- the classifier head should always train.")

    groups = [buckets[k] for k in sorted(buckets, key=lambda k: (-k[0], k[1]))]
    enc_lrs = [g["lr"] for g in groups if g["lr"] != HEAD_LR]
    print(f"  optimizer: {len(groups)} param groups | head lr {HEAD_LR:.1e} | "
          + (f"encoder lr {min(enc_lrs):.2e}..{max(enc_lrs):.2e} "
             f"({sum(len(g['params']) for g in groups if g['lr'] != HEAD_LR)} tensors)"
             if enc_lrs else "encoder fully frozen")
          + f" | wd={WEIGHT_DECAY}, "
          + f"{sum(len(g['params']) for g in groups if g['weight_decay'] == 0)} tensors exempt")
    return groups


def build_model_and_optimizer(train_df):
    """A FRESH model + optimizer per fold.

    This must not be hoisted out of the fold loop: reusing one model across folds would let fold 2
    start from weights already trained on fold 1's data — which includes fold 2's test
    participants. Every fold's score after the first would be contaminated.
    """
    model = WavLMClassifier(WAVLM_CHECKPOINT, n_classes=N_CLASSES).to(DEVICE)

    # Class weights from THIS fold's training distribution (paper: weighted CE for imbalance)
    class_counts = train_df["class"].value_counts()
    weights = torch.tensor([1.0 / class_counts[c] for c in CLASSES_SORTED], dtype=torch.float32)
    weights = weights / weights.sum() * N_CLASSES
    criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE), label_smoothing=LABEL_SMOOTHING)

    optimizer = torch.optim.AdamW(build_param_groups(model))
    return model, criterion, optimizer

In [ ]:
def run_epoch(model, criterion, optimizer, loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(train)
    for batch in loader:
        input_values = batch["input_values"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        if train:
            optimizer.zero_grad()
        logits = model(input_values, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)
    torch.set_grad_enabled(True)
    return total_loss / total, correct / total


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        logits = model(batch["input_values"].to(DEVICE),
                       attention_mask=batch["attention_mask"].to(DEVICE))
        preds.extend(logits.argmax(-1).cpu().tolist())
        labels.extend(batch["labels"].tolist())
    return preds, labels


def train_one_fold(spec):
    """Train on one fold and return its history, its test predictions, and the trained model."""
    fold = spec["fold"]
    train_loader, val_loader, test_loader = make_loaders(
        spec["train_df"], spec["val_df"], spec["test_df"])
    model, criterion, optimizer = build_model_and_optimizer(spec["train_df"])

    ckpt = WORK_DIR / f"best_wavlm_fold{fold}.pt"
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")

    stale = 0
    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, criterion, optimizer, train_loader, train=True)
        va_loss, va_acc = run_epoch(model, criterion, optimizer, val_loader, train=False)
        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss); history["val_acc"].append(va_acc)
        improved = va_loss < best_val_loss
        if improved:
            best_val_loss = va_loss
            stale = 0
            torch.save(model.state_dict(), ckpt)
        else:
            stale += 1
        print(f"  fold {fold} epoch {epoch:02d}/{N_EPOCHS} | train loss {tr_loss:.4f} "
              f"acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}"
              + ("  <- best" if improved
                 else (f"  ({stale}/{EARLY_STOP_PATIENCE})" if EARLY_STOP_PATIENCE else "")))
        # Early stopping does NOT change the reported number: the best-val-loss checkpoint is
        # what gets evaluated either way. It only stops paying for epochs spent overfitting.
        if EARLY_STOP_PATIENCE and stale >= EARLY_STOP_PATIENCE:
            print(f"  fold {fold}: early stop at epoch {epoch} — no val-loss improvement for "
                  f"{EARLY_STOP_PATIENCE} epochs (best {best_val_loss:.4f})")
            break

    # evaluate the best checkpoint, not whatever the last epoch happened to leave behind
    model.load_state_dict(torch.load(ckpt))
    preds, labels = predict_loader(model, test_loader)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro", zero_division=0)
    print(f"  fold {fold} TEST accuracy {acc:.3f} | macro-F1 {f1m:.3f}\n")
    return history, preds, labels, acc, f1m, model


if USE_CV:
    print(f"{N_FOLDS}-fold CV: training WavLM {N_FOLDS} times "
          f"({N_FOLDS} x {N_EPOCHS} = {N_FOLDS * N_EPOCHS} epochs total). This is the slow part.\n")

fold_histories, fold_scores = [], []
all_preds, all_labels = [], []          # pooled across folds -> every participant tested once
for spec in FOLDS:
    hist, preds, labels, acc, f1m, model = train_one_fold(spec)
    fold_histories.append(hist)
    fold_scores.append({"fold": spec["fold"], "accuracy": acc, "macro_f1": f1m,
                        "n_test_segments": len(labels)})
    all_preds.extend(preds); all_labels.extend(labels)

history = fold_histories[-1]            # kept for any cell expecting the single-split name
test_df = FOLDS[-1]["test_df"]          # the model left in memory is the last fold's

fold_df = pd.DataFrame(fold_scores)
if USE_CV:
    print("Per-fold results:")
    print(fold_df.round(4).to_string(index=False))
    print(f"\nAccuracy: {fold_df['accuracy'].mean():.4f} +/- {fold_df['accuracy'].std():.4f}")
    print(f"Macro-F1: {fold_df['macro_f1'].mean():.4f} +/- {fold_df['macro_f1'].std():.4f}")
    kids_per_fold = int(np.mean([f["test_df"]["participant"].nunique() for f in FOLDS]))
    print(f"\nThe +/- is the spread across folds — with ~{kids_per_fold} test participants per "
          f"fold it is usually wide, which is exactly the uncertainty a single split hides.")
    fold_df.to_csv(WORK_DIR / "cv_fold_results.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, h in enumerate(fold_histories, start=1):
    lbl = f"fold {i}" if len(fold_histories) > 1 else None
    axes[0].plot(h["train_loss"], alpha=0.8, label=f"train {lbl}" if lbl else "train")
    axes[0].plot(h["val_loss"], "--", alpha=0.8, label=f"val {lbl}" if lbl else "val")
    axes[1].plot(h["train_acc"], alpha=0.8, label=f"train {lbl}" if lbl else "train")
    axes[1].plot(h["val_acc"], "--", alpha=0.8, label=f"val {lbl}" if lbl else "val")
axes[0].set_title("Loss (solid=train, dashed=val)"); axes[0].set_xlabel("Epoch")
axes[1].set_title("Accuracy (solid=train, dashed=val)"); axes[1].set_xlabel("Epoch")
axes[0].legend(fontsize=7); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_wavlm_training_curves.png", dpi=150)
plt.show()

## 6. Test-set evaluation

Per-class precision/recall/F1, plus the two numbers a binary screening task is actually judged on:
**sensitivity** (of the children who have ASD, how many were caught) and **specificity** (of the TD
children, how many were correctly cleared). ASD is the positive class.

The paper's 0.769 is printed as a reference point only — it is a 3-class result and this is a 2-class
task, so the two are not comparable in either direction.


In [ ]:
# all_preds / all_labels were pooled across folds in the training cell above, so with USE_CV=True
# this table covers EVERY participant (each tested exactly once) rather than a single ~18-child
# test split.
test_acc = accuracy_score(all_labels, all_preds)
scope = f"pooled over {len(FOLDS)} folds — all {len(set(seg_df['participant']))} participants" \
        if USE_CV else "single held-out split"
print(f"Overall test accuracy: {test_acc:.3f}  ({scope})")
print(f"  chance for this binary task is {1 / N_CLASSES:.1%}, so the margin over chance is "
      f"{test_acc - 1 / N_CLASSES:+.3f}")
print(f"  [reference only] the paper reports 0.769 on the 3-CLASS task (ASD/ADHD/TD) from a single "
      f"split. That is a different problem with a 33.3% chance line — not a baseline this number "
      f"beats or misses.")
if USE_CV:
    print(f"  per-fold mean +/- std: {fold_df['accuracy'].mean():.3f} "
          f"+/- {fold_df['accuracy'].std():.3f}")

table3 = pd.DataFrame({
    "Precision": precision_score(all_labels, all_preds, average=None, zero_division=0),
    "Recall": recall_score(all_labels, all_preds, average=None, zero_division=0),
    "F1 Score": f1_score(all_labels, all_preds, average=None, zero_division=0),
}, index=[id2label[i] for i in sorted(id2label)]).round(2)
table3.to_csv(WORK_DIR / "table3_deep_learning_results.csv")

# Binary screening metrics. Only meaningful because the task has exactly two classes — there is no
# single sensitivity/specificity pair in the paper's 3-class setting, which is why this block is
# specific to this notebook.
POSITIVE_CLASS = "ASD"          # the condition being screened FOR
NEGATIVE_CLASS = "TD"
pos, neg = label2id[POSITIVE_CLASS], label2id[NEGATIVE_CLASS]

cm_bin = confusion_matrix(all_labels, all_preds, labels=[neg, pos])
(tn, fp), (fn, tp) = cm_bin
sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")   # recall of ASD
specificity = tn / (tn + fp) if (tn + fp) else float("nan")   # recall of TD
ppv = tp / (tp + fp) if (tp + fp) else float("nan")
npv = tn / (tn + fn) if (tn + fn) else float("nan")
balanced_acc = (sensitivity + specificity) / 2

print(f"\nBinary screening metrics (positive class = {POSITIVE_CLASS}, segment level):")
print(f"  sensitivity / recall({POSITIVE_CLASS}) : {sensitivity:.3f}   "
      f"({tp} of {tp + fn} {POSITIVE_CLASS} segments caught)")
print(f"  specificity / recall({NEGATIVE_CLASS})  : {specificity:.3f}   "
      f"({tn} of {tn + fp} {NEGATIVE_CLASS} segments correctly cleared)")
print(f"  PPV (precision, {POSITIVE_CLASS})       : {ppv:.3f}")
print(f"  NPV                          : {npv:.3f}")
print(f"  balanced accuracy            : {balanced_acc:.3f}   "
      f"(mean of the two recalls — unlike plain accuracy this is not flattered by class imbalance)")

binary_metrics = pd.Series({
    "sensitivity": sensitivity, "specificity": specificity, "PPV": ppv, "NPV": npv,
    "balanced_accuracy": balanced_acc, "accuracy": test_acc,
}).round(4)
binary_metrics.to_csv(WORK_DIR / "binary_screening_metrics.csv", header=["value"])

table3

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[id2label[i] for i in sorted(id2label)],
            yticklabels=[id2label[i] for i in sorted(id2label)], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"WavLM-Base-Plus confusion matrix — binary {' vs '.join(CLASSES_SORTED)}\n"
             f"(test acc {test_acc:.3f}, chance {1 / N_CLASSES:.1%})")
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_wavlm_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
table3.plot(kind="bar", ax=ax)
ax.set_title(f"Per-class deep learning performance — binary {' vs '.join(CLASSES_SORTED)}")
ax.set_ylabel("Score"); ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_table3_reproduction.png", dpi=150)
plt.show()


## 7. Explainability — frequency-band occlusion (reproduces Figure 2a)

Zeroes out a spectral band via a bandstop filter, re-runs the classifier, and measures the drop/gain in
predicted probability for the true class relative to the unaltered signal.


In [ ]:
from scipy.signal import butter, sosfiltfilt

FREQ_BANDS = [(100, 300), (300, 600), (600, 1000), (1000, 2000), (2000, 4000), (4000, 8000)]

def band_stop(y, sr, low, high, order=4):
    high = min(high, sr / 2 - 1)
    sos = butter(order, [low, high], btype="bandstop", fs=sr, output="sos")
    # sosfiltfilt filters forward then backward and hands back a reversed VIEW with negative
    # strides; torch.tensor() rejects those outright. ascontiguousarray makes a normal copy.
    return np.ascontiguousarray(sosfiltfilt(sos, y))

@torch.no_grad()
def predict_proba(y):
    # eval() here, not just in the test cell above: dropout is active in train mode, which makes
    # two calls on identical audio return different probabilities. The occlusion delta would then
    # include dropout noise. Setting it here makes this function deterministic on its own rather
    # than depending on whichever cell ran last.
    model.eval()
    # random_crop=False: this is called twice per measurement (clean vs occluded) and a random
    # window each time would make the delta reflect the window, not the occlusion.
    y_padded, valid_len = pad_or_crop(y, TARGET_SR, random_crop=False)
    x = torch.tensor(y_padded, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    mask = (torch.arange(x.shape[1]) < valid_len).long().unsqueeze(0).to(DEVICE)
    logits = model(x, attention_mask=mask)
    return torch.softmax(logits, dim=-1).cpu().numpy()[0]

freq_occlusion_rows = []
sample_per_class = test_df.groupby("class").head(10)  # cap for runtime; raise for a fuller sweep
for _, row in sample_per_class.iterrows():
    y, sr = librosa.load(row["segment_path"], sr=TARGET_SR, mono=True)
    true_idx = label2id[row["class"]]
    base_proba = predict_proba(y)[true_idx]
    for low, high in FREQ_BANDS:
        y_occ = band_stop(y, sr, low, high)
        occ_proba = predict_proba(y_occ)[true_idx]
        freq_occlusion_rows.append({
            "class": row["class"], "band": f"{low}-{high}", "delta_prob": base_proba - occ_proba,
        })

freq_occ_df = pd.DataFrame(freq_occlusion_rows)
freq_occ_summary = freq_occ_df.groupby(["class", "band"])["delta_prob"].mean().unstack("band")
band_order = [f"{l}-{h}" for l, h in FREQ_BANDS]
freq_occ_summary = freq_occ_summary[band_order]
freq_occ_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
freq_occ_summary.T.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Figure 2a reproduction — frequency-band occlusion sensitivity")
ax.set_ylabel(r"$\Delta$ prob (true class)")
ax.set_xlabel("Frequency band (Hz)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(WORK_DIR / "fig2a_frequency_occlusion.png", dpi=150)
plt.show()


## 8. Explainability — time occlusion (reproduces Figure 2b)

In [ ]:
def time_occlusion_curve(y, sr, true_idx, window_ms=200, max_len_s=MAX_DURATION_S):
    y, valid_len = pad_or_crop(y, sr, max_len_s, random_crop=False)
    base_proba = predict_proba(y)[true_idx]
    win = int(window_ms / 1000 * sr)
    # Only sweep windows inside the REAL audio. Occluding pure zero-padding changes nothing, and
    # averaging those no-op points into the curve flattens it toward zero.
    n_windows = max(1, int(valid_len) // win)
    deltas, times = [], []
    for i in range(n_windows):
        y_occ = y.copy()
        y_occ[i * win:(i + 1) * win] = 0.0
        occ_proba = predict_proba(y_occ)[true_idx]
        deltas.append(base_proba - occ_proba)
        times.append(i * window_ms / 1000)
    return times, deltas

time_occ_rows = []
for _, row in sample_per_class.iterrows():
    y, sr = librosa.load(row["segment_path"], sr=TARGET_SR, mono=True)
    true_idx = label2id[row["class"]]
    times, deltas = time_occlusion_curve(y, sr, true_idx)
    for t, d in zip(times, deltas):
        time_occ_rows.append({"class": row["class"], "time_s": t, "delta_prob": d})

time_occ_df = pd.DataFrame(time_occ_rows)
time_occ_curve = time_occ_df.groupby(["class", "time_s"])["delta_prob"].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
for cls in CLASSES_SORTED:
    sub = time_occ_curve[time_occ_curve["class"] == cls]
    ax.plot(sub["time_s"], sub["delta_prob"], label=cls, marker="o", markersize=3)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(r"Figure 2b reproduction — time occlusion ($\Delta$ prob over time)")
ax.set_xlabel("Time (s)"); ax.set_ylabel(r"$\Delta$ prob (true class)")
ax.legend()
plt.tight_layout()
plt.savefig(WORK_DIR / "fig2b_time_occlusion.png", dpi=150)
plt.show()


## 9. Summary

This run's numbers, with the paper's reported accuracy shown **as a labelled reference, not a
baseline**: the paper's 0.769 is a 3-class result (chance 33.3%) and everything here is 2-class
(chance 50.0%). A higher number here does not mean the reproduction beat the paper.


In [ ]:
TASK_NAME = " vs ".join(CLASSES_SORTED)
CHANCE = 1 / N_CLASSES

summary_rows = [
    {"Pipeline": "WavLM-Base-Plus — paper, 3-class ASD/ADHD/TD [reference, NOT comparable]",
     "Accuracy": 0.769, "Macro F1": np.nan, "Chance": 1 / 3},
    {"Pipeline": f"WavLM-Base-Plus — this run, binary {TASK_NAME} "
                 f"({'pooled ' + str(len(FOLDS)) + '-fold CV' if USE_CV else 'single split'})",
     "Accuracy": test_acc,
     "Macro F1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
     "Chance": CHANCE},
]
if USE_CV:
    summary_rows.append({
        "Pipeline": f"WavLM-Base-Plus — this run, binary {TASK_NAME} (mean of {len(FOLDS)} folds)",
        "Accuracy": fold_df["accuracy"].mean(),
        "Macro F1": fold_df["macro_f1"].mean(),
        "Chance": CHANCE,
    })
summary_df = pd.DataFrame(summary_rows).round(4)
# How far above its OWN chance line each row sits. This is the only column on which a 2-class and a
# 3-class result can be put side by side at all, and even then only loosely.
summary_df["Above chance"] = (summary_df["Accuracy"] - summary_df["Chance"]).round(4)
summary_df.to_csv(WORK_DIR / "final_comparison_vs_paper.csv", index=False)

print(f"This notebook's task: binary {TASK_NAME} (chance {CHANCE:.1%}).")
print("The paper's row is 3-class ASD/ADHD/TD (chance 33.3%) and is listed for reference only — "
      "do not report this run as reproducing or exceeding it.")
if USE_CV:
    print(f"Fold spread — accuracy {fold_df['accuracy'].min():.3f} to "
          f"{fold_df['accuracy'].max():.3f} (std {fold_df['accuracy'].std():.3f}). The paper's "
          f"0.769 comes from one split and carries no comparable error bar.")
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = summary_df.dropna(subset=["Accuracy"])
sns.barplot(data=plot_df, y="Pipeline", x="Accuracy", ax=ax, palette="viridis")
ax.axvline(CHANCE, color="red", ls="--", lw=1,
           label=f"chance, binary task ({CHANCE:.0%})")
ax.axvline(1 / 3, color="grey", ls=":", lw=1, label="chance, paper's 3-class task (33%)")
ax.legend(loc="lower right", fontsize=8)
ax.set_title(f"This run (binary {TASK_NAME}) vs. the paper's 3-class accuracy\n"
             f"Different tasks, different chance lines — the paper bar is a reference, not a baseline",
             fontsize=10)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_final_comparison.png", dpi=150)
plt.show()


## Notes / next steps

- **This is the binary ASD-vs-TD variant.** ADHD is removed from `CLASS_CONFIG` in Section 1, and
  Section 2 asserts the active class set is exactly `["ASD", "TD"]`, so an `SK-ADHD` folder left in
  the dataset cannot turn this back into a 3-class run by accident. For the 3-class task, use the
  3-class notebook rather than re-adding a class here.
- **Do not report this run's accuracy against the paper's 0.769.** That number is a 3-class result
  (chance 33.3%); this is 2-class (chance 50.0%). Section 6 and Section 9 keep the two labelled and
  separate, and report each against its own chance line.
- For the thesis, **sensitivity and specificity** (Section 6, ASD as the positive class) are the more
  defensible headline numbers for a screening task than accuracy — especially with unequal group
  sizes, where balanced accuracy is also printed.
- A checkpoint trained by this notebook has a **2-unit output layer**. The inference notebook
  (`kaggle_inference.ipynb`) must be set to `CLASS_SET = "ASD_TD"` to load it; a 3-class setting
  raises rather than mislabelling.
- The deep-learning section only — the acoustic-feature + classical-ML pipeline (Table 1, Table 2) is
  not included here.
- All generated tables/figures are saved under `/kaggle/working/` for direct inclusion in the thesis.
